# Exercise 3.1: Your First LangChain Chain

**Module:** 3 — LangChain Fundamentals
**Level:** Basic

In this notebook, you'll send your first message to an LLM, then learn how to connect components together into a **chain** — the core building block of LangChain.

No prior AI experience needed. We start from zero.

**What you'll do:**
1. Send a message to an LLM and get a response
2. Use a prompt template (like f-strings, but smarter)
3. Connect prompt → model → output parser into a chain
4. Build a multi-step chain that processes data through stages

## 1. Setup

First, install the packages and set your API key.

Get your free Groq API key at: https://console.groq.com/keys

In [ ]:
# Install LangChain and the Groq integration
# langchain-groq lets LangChain talk to Groq's free API
!pip install langchain langchain-groq -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. Your First LLM Call

The simplest thing you can do: send a message, get a response.

We're using **Groq**, which runs open-source LLMs (like Llama 3.3) for free. Think of it as a fast, free alternative to OpenAI.

In [ ]:
from langchain_groq import ChatGroq
# ChatGroq is LangChain's wrapper for Groq's API.
# It gives us a standard interface — same code works
# if we switch to OpenAI, Anthropic, or any other provider.

# Create the model
# temperature=0 means deterministic (same input → same output)
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Send a simple message
response = model.invoke("What is the IATA code for Istanbul airport?")

# The response is an AIMessage object, not just a string
print(response)
print(f"\nAnswer: {response.content}")
print(f"Type: {type(response)}")

That's it. One line to create the model, one line to call it.

But this is just a chatbot — we send a question, we get an answer. To build something useful, we need **structure**.

## 3. Prompt Templates

Hardcoding questions is like hardcoding SQL queries — it works but it's fragile.

A **prompt template** is like an f-string with superpowers: it has variables you fill in at runtime.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# ChatPromptTemplate creates reusable prompt structures.
# You define the shape once, fill in variables later.

# Define a template with two variables: {origin} and {destination}
prompt = ChatPromptTemplate.from_messages([
    # "system" sets the AI's role and behavior
    ("system", "You are a helpful travel assistant. Be concise."),
    # "human" is the user's message — variables go in {curly braces}
    ("human", "What airlines fly from {origin} to {destination}?")
])

# Fill in the variables and see what the prompt looks like
filled = prompt.invoke({"origin": "Istanbul", "destination": "Tokyo"})
print(filled)

Now we can reuse this template for any city pair. But we're still doing two separate steps — filling the template, then calling the model. Let's connect them.

## 4. Chains: Connecting Components

A **chain** connects components together with the **pipe operator** `|`.

If you've used Unix/Linux, you know the concept:
```
cat log.txt | grep ERROR | wc -l
```

Same idea. Output of one step flows into the next:
```
prompt | model | parser
```

- `prompt` takes your variables, produces formatted messages
- `model` takes messages, produces an AI response
- `parser` takes the response, extracts just the text

In [ ]:
from langchain_core.output_parsers import StrOutputParser
# StrOutputParser extracts just the text content from the AI response.
# Without it, you get an AIMessage object. With it, you get a clean string.

# Build the chain: prompt → model → parser
# The | operator connects them — output of each step feeds into the next
chain = prompt | model | StrOutputParser()

# Run the chain — just pass the variables, everything flows automatically
result = chain.invoke({"origin": "Istanbul", "destination": "Tokyo"})

print(result)
print(f"\nType: {type(result)}")  # Now it's a plain string, not AIMessage

**That's a chain.** Three components, one pipe. Data flows through automatically.

This is the core pattern you'll use throughout the entire training.

## 5. Multi-Step Chain

Chains become powerful when you connect multiple steps. Here's a real example:

1. **Step 1:** Analyze a flight disruption
2. **Step 2:** Generate a passenger notification based on the analysis

In [ ]:
# Step 1: Analyze the disruption
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a flight operations analyst. Analyze disruptions concisely."),
    ("human", "Flight {flight} from {origin} to {dest} is delayed {delay_min} minutes. "
             "Classify severity (minor/moderate/severe) and list 2 recommended actions.")
])

# This chain produces the analysis as text
analysis_chain = analysis_prompt | model | StrOutputParser()

# Test it
analysis = analysis_chain.invoke({
    "flight": "TK1234",
    "origin": "IST",
    "dest": "CDG",
    "delay_min": 180
})

print("=== ANALYSIS ===")
print(analysis)

In [ ]:
# Step 2: Generate passenger notification based on the analysis
notification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write short, empathetic passenger notifications. Max 3 sentences."),
    ("human", "Based on this analysis, write an SMS to affected passengers:\n\n{analysis}")
])

notification_chain = notification_prompt | model | StrOutputParser()

# Feed the analysis from Step 1 into Step 2
sms = notification_chain.invoke({"analysis": analysis})

print("=== PASSENGER SMS ===")
print(sms)

Two chains, two steps: analyze → notify. Each chain does one thing well.

In Module 4, you'll learn how to connect these into a **graph** where chains can branch, loop, and make decisions.

---

## YOUR TURN: Exercise A

Build a chain that takes a city name and returns the top 3 things to do there.

Hint: create a prompt template with a `{city}` variable, connect it to the model and a parser.

In [ ]:
# YOUR CODE HERE
# 1. Create a prompt template with a {city} variable
# 2. Build a chain: prompt | model | StrOutputParser()
# 3. Test with: chain.invoke({"city": "Paris"})


## YOUR TURN: Exercise B

Extend Exercise A into a two-step chain:
1. Get top 3 things to do in a city
2. Generate a 1-day itinerary from those activities

Hint: run the first chain, pass its output as a variable to the second chain.

In [ ]:
# YOUR CODE HERE
# Step 1: activities_chain = ...
# Step 2: itinerary_chain = ...
# Run both and print the itinerary


## Key Takeaways

- **LLM call:** `model.invoke("question")` — simplest interaction
- **Prompt template:** Reusable prompt with `{variables}` — like f-strings for AI
- **Chain:** `prompt | model | parser` — components connected with pipe `|`
- **Multi-step:** Feed one chain's output into another chain's input

**Next:** In Exercise 3.2, you'll add **memory** so your chains can remember previous conversations.